In [141]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [142]:
df = pd.read_csv(r'R:\PROJECTS\Public_Transport_Delay_Prediction\public_transport_delays.csv')
df.head()

,trip_id,date,time,transport_type,route_id,origin_station,destination_station,scheduled_departure,scheduled_arrival,actual_departure_delay_min,...,wind_speed_kmh,precipitation_mm,event_type,event_attendance_est,traffic_congestion_index,holiday,peak_hour,weekday,season,delayed
0,T00000,2023-01-01,05:00:00,Tram,Route_15,Station_31,Station_6,05:02:00,05:55:00,12,...,46,13.0,NaN,500,81,0,1,6,Winter,0
1,T00001,2023-01-01,05:15:00,Metro,Route_12,Station_49,Station_32,05:16:00,05:55:00,15,...,11,11.4,NaN,0,53,0,0,6,Autumn,1
2,T00002,2023-01-01,05:30:00,Bus,Route_16,Station_29,Station_42,05:33:00,06:17:00,0,...,31,14.1,Sports,0,67,1,0,6,Autumn,0
3,T00003,2023-01-01,05:45:00,Tram,Route_19,Station_26,Station_18,05:49:00,06:08:00,15,...,41,6.4,NaN,500,84,0,0,6,Winter,1
4,T00004,2023-01-01,06:00:00,Tram,Route_8,Station_18,Station_15,06:00:00,06:35:00,-1,...,30,18.5,NaN,500,46,0,0,6,Spring,1


In [143]:
df.columns

Index(['trip_id', 'date', 'time', 'transport_type', 'route_id',
       'origin_station', 'destination_station', 'scheduled_departure',
       'scheduled_arrival', 'actual_departure_delay_min',
       'actual_arrival_delay_min', 'weather_condition', 'temperature_C',
       'humidity_percent', 'wind_speed_kmh', 'precipitation_mm', 'event_type',
       'event_attendance_est', 'traffic_congestion_index', 'holiday',
       'peak_hour', 'weekday', 'season', 'delayed'],
      dtype='str')

In [144]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   trip_id                     2000 non-null   str    
 1   date                        2000 non-null   str    
 2   time                        2000 non-null   str    
 3   transport_type              2000 non-null   str    
 4   route_id                    2000 non-null   str    
 5   origin_station              2000 non-null   str    
 6   destination_station         2000 non-null   str    
 7   scheduled_departure         2000 non-null   str    
 8   scheduled_arrival           2000 non-null   str    
 9   actual_departure_delay_min  2000 non-null   int64  
 10  actual_arrival_delay_min    2000 non-null   int64  
 11  weather_condition           2000 non-null   str    
 12  temperature_C               2000 non-null   float64
 13  humidity_percent            2000 non-null   

In [145]:
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

In [146]:
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S')
df['hour'] = pd.to_datetime(df['time'], format='%H:%M').dt.hour

In [147]:
df['scheduled_arrival'] = pd.to_datetime(df['scheduled_arrival'], format='%H:%M:%S')
df['scheduled_departure'] = pd.to_datetime(df['scheduled_departure'], format='%H:%M:%S')
df['scheduled_arrival_hour'] = pd.to_datetime(df['scheduled_arrival'], format='%H:%M').dt.hour
df['scheduled_departure_hour'] = pd.to_datetime(df['scheduled_departure'], format='%H:%M').dt.hour
df['scheduled_arrival_minute'] = pd.to_datetime(df['scheduled_arrival'], format='%H:%M').dt.minute
df['scheduled_departure_minute'] = pd.to_datetime(df['scheduled_departure'], format='%H:%M').dt.minute


In [148]:
df.isna().sum()


trip_id                          0
date                             0
time                             0
transport_type                   0
route_id                         0
origin_station                   0
destination_station              0
scheduled_departure              0
scheduled_arrival                0
actual_departure_delay_min       0
actual_arrival_delay_min         0
weather_condition                0
temperature_C                    0
humidity_percent                 0
wind_speed_kmh                   0
precipitation_mm                 0
event_type                    1173
event_attendance_est             0
traffic_congestion_index         0
holiday                          0
peak_hour                        0
weekday                          0
season                           0
delayed                          0
month                            0
day                              0
hour                             0
scheduled_arrival_hour           0
scheduled_departure_

In [149]:
df.duplicated().sum()

np.int64(0)

In [150]:
df['event_type'] = df['event_type'].fillna('No Event')

In [151]:
cal_col = df.select_dtypes(include=["object", "string"]).columns

print("Categorical Columns:", cal_col)

Categorical Columns: Index(['trip_id', 'transport_type', 'route_id', 'origin_station',
       'destination_station', 'weather_condition', 'event_type', 'season'],
      dtype='str')


In [152]:
num_col = df.select_dtypes(include=["int64", "float64"]).columns
#print("Numerical Columns:", num_col)
num_col = num_col.drop(['actual_arrival_delay_min','delayed'])


In [153]:
print('Before Scaling:')
df[num_col]

Before Scaling:


,actual_departure_delay_min,temperature_C,humidity_percent,wind_speed_kmh,precipitation_mm,event_attendance_est,traffic_congestion_index,holiday,peak_hour,weekday
0,12,5.1,52,46,13.0,500,81,0,1,6
1,15,34.0,64,11,11.4,0,53,0,0,6
2,0,29.5,35,31,14.1,0,67,1,0,6
3,15,27.4,55,41,6.4,500,84,0,0,6
4,-1,0.1,90,30,18.5,500,46,0,0,6
...,...,...,...,...,...,...,...,...,...,...
1995,15,17.2,98,35,4.6,0,96,0,0,5
1996,11,0.0,89,44,15.4,0,12,0,1,6
1997,1,12.9,95,32,2.7,0,24,1,0,6
1998,7,17.8,55,35,8.8,2000,23,0,0,6


In [154]:
df[cal_col]

,trip_id,transport_type,route_id,origin_station,destination_station,weather_condition,event_type,season
0,T00000,Tram,Route_15,Station_31,Station_6,Storm,No Event,Winter
1,T00001,Metro,Route_12,Station_49,Station_32,Rain,No Event,Autumn
2,T00002,Bus,Route_16,Station_29,Station_42,Clear,Sports,Autumn
3,T00003,Tram,Route_19,Station_26,Station_18,Clear,No Event,Winter
4,T00004,Tram,Route_8,Station_18,Station_15,Snow,No Event,Spring
...,...,...,...,...,...,...,...,...
1995,T01995,Bus,Route_11,Station_46,Station_39,Storm,No Event,Winter
1996,T01996,Train,Route_9,Station_44,Station_42,Snow,Festival,Winter
1997,T01997,Bus,Route_12,Station_4,Station_45,Snow,No Event,Summer
1998,T01998,Tram,Route_17,Station_29,Station_48,Clear,No Event,Summer


In [155]:
df = df.drop(columns=['trip_id', 'date', 'delayed','time','scheduled_arrival','scheduled_departure'])

In [156]:
low_cardinality = [
    'transport_type',
    'weather_condition',
    'event_type',
    'season'
]

high_cardinality = [
    'route_id',
    'origin_station',
    'destination_station'
]

df = pd.get_dummies(
    df,
    columns=low_cardinality,
    drop_first=True
)

le = LabelEncoder()

for col in high_cardinality:
    df[col] = le.fit_transform(df[col])

In [157]:
df.columns

Index(['route_id', 'origin_station', 'destination_station',
       'actual_departure_delay_min', 'actual_arrival_delay_min',
       'temperature_C', 'humidity_percent', 'wind_speed_kmh',
       'precipitation_mm', 'event_attendance_est', 'traffic_congestion_index',
       'holiday', 'peak_hour', 'weekday', 'month', 'day', 'hour',
       'scheduled_arrival_hour', 'scheduled_departure_hour',
       'scheduled_arrival_minute', 'scheduled_departure_minute',
       'transport_type_Metro', 'transport_type_Train', 'transport_type_Tram',
       'weather_condition_Cloudy', 'weather_condition_Fog',
       'weather_condition_Rain', 'weather_condition_Snow',
       'weather_condition_Storm', 'event_type_Festival', 'event_type_No Event',
       'event_type_Parade', 'event_type_Protest', 'event_type_Sports',
       'season_Spring', 'season_Summer', 'season_Winter'],
      dtype='str')

In [158]:
X = df.drop(columns=['actual_arrival_delay_min'])
y = df['actual_arrival_delay_min']
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=40,test_size=0.2)

In [163]:
X_train[num_col] = StandardScaler().fit_transform(X_train[num_col])
X_test[num_col] = StandardScaler().fit_transform(X_test[num_col])

In [164]:
rf_reg = RandomForestRegressor(n_estimators=100, random_state=40)

rf_reg.fit(X_train, y_train)
pred = rf_reg.predict(X_test)
pred

array([14.98, 14.49, 14.06, 13.29, 10.44, 13.24, 12.87, 13.12, 13.35,
       12.46, 13.37, 14.63, 11.23, 14.86, 12.72, 13.75, 11.66, 14.11,
       15.94, 13.36, 16.19, 14.94, 13.47, 10.82, 12.45, 12.99, 13.18,
       12.71, 15.15, 11.21, 10.97, 15.34, 13.79, 12.77, 13.35, 12.98,
       12.85, 14.94, 12.14, 11.02, 12.05, 11.84, 15.35, 10.53, 13.98,
        9.83, 15.62, 15.25, 11.74, 14.07, 12.93, 13.6 , 14.51, 11.25,
       12.8 , 11.28, 13.52, 11.1 , 14.46, 11.88, 14.38, 13.64, 13.07,
       13.58, 15.35, 11.57, 12.78, 13.93, 14.18, 12.93, 12.81, 13.11,
       11.61, 13.28, 12.74,  9.33, 12.03, 11.77, 12.49, 13.26, 12.97,
       15.86, 11.91, 10.34, 10.1 , 14.13, 12.44, 13.35,  9.71, 10.95,
       14.  , 12.85, 12.32, 10.71, 13.25, 12.92, 13.52, 10.96, 15.36,
       12.64, 12.21, 14.04, 14.97, 11.99, 12.72, 12.23, 15.97, 12.16,
       13.31, 13.3 , 12.55, 10.07, 14.44, 14.6 , 12.87, 10.5 , 13.49,
       12.15, 14.2 , 10.67, 11.7 , 12.08, 12.38, 11.16, 13.52, 11.68,
       10.47, 10.64,

In [165]:
print("Mean Squared Error:", mean_squared_error(y_test, pred))
print("R-squared:", r2_score(y_test, pred))



Mean Squared Error: 86.87183925000001
R-squared: -0.043321428777848725


In [162]:
cv_scores = cross_val_score(rf_reg, X, y, cv=5, scoring='r2')
mean_cv_score = cv_scores.mean()
print("Cross-Validation R-squared Scores:", mean_cv_score)

Cross-Validation R-squared Scores: -0.04426057354888076
